In [ ]:
## Nikolay Vorontsov, 23.11.2024, Prompting LLMs with validation set, ask to find hallucinations and label them.
## Model used: gemini-1.5-flash
## required files: llabel_validation_set_with_llm_for_baseline.env
##                 mushroom.en-val.v2.unlabeled.jsonl

In [ ]:
#INSTALL DEPENDENCIES

!pip install openai

In [1]:
# IMPORT LIBRARIES

import configparser
import json
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

import time
from openai import OpenAI


Mounted at /content/drive


In [2]:
# DEFINE VARIABLES
client = OpenAI(api_key = userdata.get('GPT_MUSHROOM'))
model_used = "gpt-4o-mini" #"gpt-4o-2024-08-06"

prompts = configparser.ConfigParser()

prompts.read('/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_GPT_v2.env')
output_file=f"/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_{model_used}_v2.jsonl"

set_to_label =  '/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/mushroom.en-val.v2.unlabeled.jsonl'

In [3]:
# "a" creates the file if it doesn't exist
with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gpt-4o-mini_v2.jsonl is created or already exists.


In [4]:
# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(set_to_label)


In [ ]:
#FUNCTIONS

In [5]:
# COMPILE PROMPTS
def define_prompt(datapoint):

  #samples can be also imported from a jsonl file.
  Sample1 = prompts.get('SAMPLES', 'Sample1')

  prompt1 = (
      f"{prompts.get('PROMPTS', 'p0')}"
      f"{datapoint}"
      f"{prompts.get('PROMPTS', 'p1')}"
      f"{prompts.get('PROMPTS', 'p2')}"
      f"{prompts.get('PROMPTS', 'p3')}"
      f"{prompts.get('PROMPTS', 'p4')}"
      f"{prompts.get('PROMPTS', 'p5')}"
      )

  return prompt1

In [6]:
define_prompt(data[1])

'This json line contains output from an llm, question-answer pair in "model_input", "model_output_text".{\'id\': \'val-en-2_unlabeled\', \'lang\': \'EN\', \'model_input\': \'How many genera does the Erysiphales order contain?\', \'model_output_text\': \'The Elysiphale order contains 5 genera.\', \'model_id\': \'tiiuae/falcon-7b-instruct\', \'soft_labels\': [], \'hard_labels\': [], \'model_output_logits\': [-6.199614048, -13.7564926147, -14.0058326721, -17.6579284668, -12.7987499237, -9.0510673523, -7.8389821053, -11.1029033661, -8.5361289978, -11.487569809, -8.2251300812, -9.1043262482], \'model_output_tokens\': [\'The\', \'ĠE\', \'lys\', \'iph\', \'ale\', \'Ġorder\', \'Ġcontains\', \'Ġ\', \'5\', \'Ġgenera\', \'.\', \'<|endoftext|>\']}Find all possible hallucinations (words/parts of text) in the "model_output_text", based on the knowledge from the internet. Select minimal hallucinations for the text to be correct.List each hallucination on a separate line. Do not output parts of the te

In [7]:
def process_data(datapointX):
    selected_prompt = define_prompt(datapointX)
    completion = client.chat.completions.create(
        model=f"{model_used}",
        messages=[{"role": "user", "content": selected_prompt}]
        )
    hallucinated_words = [list_element.strip("- ") for list_element in completion.choices[0].message.content.split("\n") if list_element] ## this ensure list_element is not empty
    return hallucinated_words

In [8]:
## TEST process_data(datapoint)
process_data(data[11])

['Salzberg', 'Red bull']

In [9]:
def find_spans(datapointX, hallucinated_words):

  spans = []
  model_output_text = datapointX["model_output_text"]

  for word in hallucinated_words:
      # Initialize the starting index for each word search
      start_index = 0
      while True:
          start_index = model_output_text.find(word, start_index)
          if start_index == -1:
              break
          end_index = start_index + len(word)
          spans.append([start_index, end_index])
          # Move the starting index past the current word to avoid overlapping results
          start_index = end_index

  return spans


In [10]:
## Assessing the output file

def last_line_number(file_path):
  # Read the last line of the file
  last_line = None
  with open(file_path, "r") as file:
      for line in file:
          last_line = line.strip()  # Store the current line

  # Parse the JSON object from the last line
  if last_line:
      last_data = json.loads(last_line)
      #print("Last JSON object:", last_data)
      return last_data["number"]
  else:
      print("The file is empty")
      return None


In [11]:
##TESTING
last_line_number(output_file)

The file is empty


In [12]:
for key, value in data[0].items():
  print(f'"{key}":') #type(value))

"id":
"lang":
"model_input":
"model_output_text":
"model_id":
"soft_labels":
"hard_labels":
"model_output_logits":
"model_output_tokens":


In [13]:
def label_and_save_data(data):
    processed_count = 0  # Counter for newly processed entries

    for number, datapoint in enumerate(data, start=1):
        # Get the last processed ID from the output file
        last_processed_number = last_line_number(output_file) or 0
        #print("Last processed ID:", last_processed_number)

        # Skip already processed entries
        if number <= last_processed_number:
            continue

        # Define prompt and process data
        prompt = define_prompt(datapoint)
        hallucinated_words = process_data(datapoint)
        hard_labels = find_spans(datapoint, hallucinated_words)

        # Save the datapoint to the JSONL file
        with open(output_file, "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "number": number,
                "id": datapoint["id"],
                "lang": datapoint["lang"],
                "model_input": datapoint["model_input"],
                "model_output_text": datapoint["model_output_text"],
                "model_id": datapoint["model_id"],
                "hallucinated_words": hallucinated_words,
                "soft_labels": [],
                "hard_labels": hard_labels,
                "model_output_logits": datapoint["model_output_logits"],
                "model_output_tokens": datapoint["model_output_tokens"],
            }

            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")
        print(datapoint["model_output_text"])
        print(hallucinated_words)
        print(hard_labels)
        print("-------------------------------")

        # Increment the processed count
        processed_count += 1

        # Stop processing after 5 new entries, wait for 10 seconds
        #if processed_count >= 5:
        #    print(f"Processed {processed_count} entries. Waiting for 10 sec. at ID {number}.")
        #    time.sleep(10)

        #    processed_count = 0


In [14]:
label_and_save_data(data)

The file is empty
Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
['Petra van Stoveren', 'silver', '2008', 'Summer Olympics', 'Beijing', 'China']
[[0, 18], [25, 31], [45, 49], [50, 65], [69, 76], [78, 83]]
-------------------------------
The Elysiphale order contains 5 genera.
['Elysiphale', '5']
[[4, 14], [30, 31]]
-------------------------------
Yes, all arachnids have antennas. However, not all of them are visible to the naked eye.
['all arachnids have antennas', 'not all of them are visible to the naked eye']
[[5, 32], [43, 87]]
-------------------------------
Chance the rapper debuted in 2011.
['201', '1']
[[29, 32], [31, 32], [32, 33]]
-------------------------------
The UN's Sustainable City initiative defines a city as one that is:
- Equipped with infrastructure and services to ensure sustainable and equitable access to a range of basic services, such as water, sanitation, and electricity;
-.
["UN's Sustainable City initiative defines a city

In [15]:
output_file

'/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_validation_set_with_llm_for_baseline_outputs/mushroom.en-val.v2.unlabeled.labelled_with_gpt-4o-mini_v2.jsonl'

In [18]:
## Resave a copy with no extra keys.

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')
        print(data_to_resave['id'])

        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels'] if label]
        print(soft_labels)

        # Save the datapoint to the JSONL file
        with open(f"mushroom.en-val.v2.unlabeled.labelled_with_{model_used}_no_extra_keys_soft_labels_prob1_v2.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")



val-en-1
[[0, 18], [25, 31], [45, 49], [50, 65], [69, 76], [78, 83]]
[{'start': 0, 'prob': 1.0, 'end': 18}, {'start': 25, 'prob': 1.0, 'end': 31}, {'start': 45, 'prob': 1.0, 'end': 49}, {'start': 50, 'prob': 1.0, 'end': 65}, {'start': 69, 'prob': 1.0, 'end': 76}, {'start': 78, 'prob': 1.0, 'end': 83}]
val-en-2
[[4, 14], [30, 31]]
[{'start': 4, 'prob': 1.0, 'end': 14}, {'start': 30, 'prob': 1.0, 'end': 31}]
val-en-3
[[5, 32], [43, 87]]
[{'start': 5, 'prob': 1.0, 'end': 32}, {'start': 43, 'prob': 1.0, 'end': 87}]
val-en-4
[[29, 32], [31, 32], [32, 33]]
[{'start': 29, 'prob': 1.0, 'end': 32}, {'start': 31, 'prob': 1.0, 'end': 32}, {'start': 32, 'prob': 1.0, 'end': 33}]
val-en-5
[[4, 67], [70, 228], [230, 231]]
[{'start': 4, 'prob': 1.0, 'end': 67}, {'start': 70, 'prob': 1.0, 'end': 228}, {'start': 230, 'prob': 1.0, 'end': 231}]
val-en-6
[[0, 7], [101, 109], [175, 183], [238, 246], [302, 308], [345, 349], [349, 354]]
[{'start': 0, 'prob': 1.0, 'end': 7}, {'start': 101, 'prob': 1.0, 'end': 